# 6주차 실습. 8개 군집 고리로 보는 GAN 과 WGAN-GP
*Week 6 lab. A GAN and a WGAN-GP on a ring of eight clusters*

**제출하지 않습니다. 채점하지 않습니다.**
*Nothing to submit, nothing graded.*

**이 실습의 목표 한 문장.** 평범한 GAN 이 고리의 군집 일부를 놓치는 것을 손실 곡선이 아니라 표본으로 확인하고, D 의 손실에 `.detach()` 가 왜 필요한지 두 줄을 직접 써서 확인한 뒤, 같은 시작 시드에서 WGAN-GP 로 바꾸면 무엇이 달라지는지 셉니다(한 가지만 바꾼 비교는 아닙니다).
*Goal in one sentence. See in the samples, not in the loss curves, that a plain GAN can miss clusters of a ring; write the two loss lines yourself and see why D's loss needs `.detach()`; then switch to WGAN-GP on the same starting seed and count again (not a one-change comparison).*

| 절 / Section | 하는 일 / What you do | 시간 / Time |
|---|---|---|
| §0 | 준비. 쉬는 시간에 실행하세요(MNIST 내려받기) / Setup. Run it during the break (downloads MNIST) | 1분 / 1 min |
| §1 | 예측 세 개를 적고, GAN 을 돌려 군집 세기 / Write three predictions, run a GAN, count clusters | 10분 / 10 min |
| §2 | 빈칸 2개, d_loss 와 g_loss. 내 두 줄로 다시 돌리기 / Two blanks, d_loss and g_loss. Re-run with your lines | 15분 / 15 min |
| §3 | WGAN-GP 로 바꾸기. 같은 고리, 같은 시작 시드 / Switch to WGAN-GP. Same ring, same starting seed | 12분 / 12 min |
| §4 | (선택) MNIST 에서 작은 DCGAN / (Optional) a small DCGAN on MNIST | CPU 약 2분 / about 2 min on a CPU |
| §5 | 예측 다시 읽기 / Re-read your predictions | 5분 / 5 min |

수업 중 실습은 45분입니다. §4 는 시간이 남을 때만 하세요.
*The lab gets 45 minutes in class. Do §4 only if time is left.*

슬라이드와 짝(6주차 본 덱): §1 「New Problem ①  G Covers Only Part of the Data」 · 「New Problem ②  G and D Keep Chasing Each Other」, §2 「From D's Raw Score to a Probability」 · 「D's Loss Is Binary Cross-Entropy」 · 「What Loss Do We Actually Use for G?」, §3 「Closer, but D Still Says “Generated”」 · 「A Score That Changes With Every Step Closer」 · 「D Becomes a Critic」 · 「Optional G · Why the Critic's Slope Must Be Limited」 · 「Optional H · The Penalty That Keeps Slopes Near 1」, §4 「DCGAN Is a GAN Built for Images」 · 「Stride and Strided Convolution」 · 「Transposed Convolution」 · 「Did We Get Sharper Samples?」, §5 「Training Well ≠ Generating Well」, 함정 「Four Ways to Get This Wrong」.
*Matching slides in the Week 6 main deck, section by section, with the same titles.*

**GPU 가 없어도 됩니다.** §1 ~ §3 은 2코어 CPU 에서 각각 1분 안팎입니다. §4 만 GPU 에서 훨씬 빠릅니다.
*No GPU needed. Sections 1 to 3 take about a minute each on a 2-core CPU. Only §4 is much faster on a GPU.*

## §0 준비 / Setup
아래 셀을 실행하면 라이브러리를 불러오고, 고리 데이터와 그림 도구를 정의하고, §4 에서 쓸 MNIST 를 내려받습니다(약 30초). **쉬는 시간에 실행해 두세요.** 내려받기가 실패해도 §1 ~ §3 은 그대로 진행됩니다(§4 에서 다시 시도).
*Run the cell below. It imports the libraries, defines the ring data and the plotting helpers, and downloads MNIST for §4 (about 30 seconds). **Run it during the break.** If the download fails, §1 to §3 still run; §4 tries again.*

먼저 **파일 → Drive에 사본 저장**을 누르세요. 그러지 않으면 작업 내용이 사라집니다.
*First, File → Save a copy in Drive. Otherwise your work is lost.*

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

torch.set_num_threads(2)   # 스레드 수 고정. 같은 컴퓨터에서 다시 돌리면 같은 숫자가 나온다 / fix the thread count so a re-run gives the same numbers
device = "cuda" if torch.cuda.is_available() else "cpu"   # §4 에서만 씀. GPU 가 있으면 GPU / used only in §4; GPU if available
print("§4 장치 / §4 device:", device)

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 바로 아래 'Solution / 정답' 셀을 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell just below and continue.\n"
)

# ---- 고리 위 8개 군집 / Eight clusters on a ring ----
R, STD = 2.0, 0.05          # 반지름 2 인 원 위, 군집마다 표준편차 0.05 / on a circle of radius 2, standard deviation 0.05 per cluster
Z_DIM, B = 16, 256          # 잡음 z 의 차원, 한 걸음에 쓰는 점의 수 / dimension of the noise z, points per step
CENTERS = torch.tensor([[R * math.cos(2 * math.pi * k / 8), R * math.sin(2 * math.pi * k / 8)]
                        for k in range(8)])   # 군집 중심 8개. 크기 (8, 2) / the 8 cluster centers, shape (8, 2)

def sample_ring(n):
    """진짜 점 n 개. 군집을 하나 고르고 그 중심 둘레에 작은 잡음을 더한다.
    n real points. Pick a cluster, then add small noise around its center."""
    k = torch.randint(0, 8, (n,))              # 군집 번호 0 ~ 7 을 무작위로 / cluster index 0 to 7 at random
    return CENTERS[k] + STD * torch.randn(n, 2)

def make_G():
    """생성자 G. 잡음 z(숫자 16개)를 점 하나(숫자 2개)로 바꾼다.
    Generator G. Turns noise z (16 numbers) into one point (2 numbers)."""
    return nn.Sequential(nn.Linear(Z_DIM, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 2))

def make_D():
    """판별자 D. 점 하나를 받아 숫자 하나를 낸다. 시그모이드는 bce 안에 있다(아래).
    Discriminator D. Takes one point, returns one number. The sigmoid lives inside bce (below)."""
    return nn.Sequential(nn.Linear(2, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 1))

Z_FIX = torch.randn(1000, Z_DIM, generator=torch.Generator().manual_seed(1234))   # 그림마다 같은 z 1,000개 / the same 1,000 z for every picture

def coverage(G):
    """고정 z 1,000개를 G 에 넣고, 군집마다 중심에서 0.2(표준편차 4배) 안에 떨어진 점을 센다.
    50개(5%) 이상이면 그 군집을 「덮었다」고 센다. 반환: 덮은 군집 수, 군집별 점 수, 점 1,000개.
    Feed the 1,000 fixed z to G and count, per cluster, the points within 0.2 (4 standard deviations) of its center.
    A cluster with 50 or more points (5%) counts as covered. Returns: clusters covered, points per cluster, the 1,000 points."""
    with torch.no_grad():                      # 그림만 그리므로 기울기는 필요 없다 / only for a picture, no gradients needed
        x = G(Z_FIX)
    d = torch.cdist(x, CENTERS)                # 점과 중심 8개 사이 거리. 크기 (1000, 8) / distance to the 8 centers, shape (1000, 8)
    m = d.min(1)                               # 가장 가까운 중심과 그 거리 / nearest center and its distance
    near = m.values < 4 * STD                  # 그 거리가 0.2 보다 작은가 / closer than 0.2?
    counts = torch.bincount(m.indices[near], minlength=8)   # 군집마다 센 점의 수 / points counted per cluster
    return int((counts >= 50).sum()), counts.tolist(), x.numpy()

def off_cluster(counts):
    """군집 밖 비율. 1,000개 가운데 어느 군집 중심에서도 0.2 안에 들지 못한 점의 비율.
    덮은 군집 수가 놓치는 것, 곧 군집 사이에 흩어진 점을 봅니다.
    Off-cluster share: of the 1,000 points, the share not within 0.2 of any cluster center.
    It shows what the covered count misses: points scattered between the clusters."""
    return 1 - sum(counts) / len(Z_FIX)

GREEN, GOLD = "#406551", "#C9971C"           # 슬라이드와 같은 색 / the slides' colors
def show_rings(snaps, label):
    """스냅숏마다 점(금색)과 군집 원(초록, 반지름 0.2)을 그린다. 그림 글자는 영어만.
    For each snapshot, draw the points (gold) and the cluster circles (green, radius 0.2). English only in figures."""
    steps = sorted(snaps)
    fig, axes = plt.subplots(1, len(steps), figsize=(2.7 * len(steps), 2.9))
    for ax, s in zip(np.atleast_1d(axes), steps):
        n_cov, counts, x = snaps[s]
        ax.scatter(x[:, 0], x[:, 1], s=3, c=GOLD, alpha=0.6, linewidths=0)
        for cx, cy in CENTERS.tolist():
            ax.add_patch(plt.Circle((cx, cy), 4 * STD, fill=False, ec=GREEN, lw=1.2))
        ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6); ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"{label}, step {s:,}\n{n_cov} of 8 covered, {off_cluster(counts):.0%} off-cluster", fontsize=9)
    plt.tight_layout(); plt.show()

def bce(logits, target):
    """이진 교차엔트로피(BCE). 코드의 D 는 시그모이드 앞의 값(로짓)을 내고, 슬라이드의 확률 D(x) 는 sigmoid(로짓)이다.
    bce(로짓, 1) = -log D(x),  bce(로짓, 0) = -log(1 - D(x)). target 은 모든 행에 같은 1 또는 0.
    Binary cross-entropy. In code, D outputs the value before the sigmoid (a logit); the slides' probability D(x) is sigmoid(logit).
    bce(logit, 1) = -log D(x),  bce(logit, 0) = -log(1 - D(x)). target is 1 or 0, the same for every row."""
    return F.binary_cross_entropy_with_logits(logits, torch.full_like(logits, float(target)))

# MNIST 는 §4 에서만 씁니다. 지금 내려받아 둡니다(약 10 MB) / MNIST is used only in §4; download it now (about 10 MB)
try:
    mnist = datasets.MNIST("data", train=True, download=True, transform=transforms.ToTensor())
    print("MNIST 학습 이미지 / MNIST training images:", len(mnist))
except Exception as e:                       # 네트워크가 막혀도 §1 ~ §3 은 진행 / §1 to §3 do not need MNIST
    mnist = None
    print("MNIST 를 내려받지 못했습니다. §1 ~ §3 은 그대로 진행하세요. §4 에서 다시 시도합니다.")
    print("Could not download MNIST. Go on with §1 to §3; §4 tries again.", type(e).__name__)
# 기대값 / Expected   60000

## §1 먼저 예측하기 / Predict first
코드를 돌리기 **전에** 아래 세 질문의 답을 한 줄씩 적어 두세요. 틀려도 괜찮습니다. §5 에서 다시 읽습니다.
*Before running anything, write a one-line answer to each question. Being wrong is fine. You will re-read them in §5.*

진짜 데이터는 반지름 2 인 원 위의 군집 8개입니다(아래 그림의 초록 원).
*The real data are 8 clusters on a circle of radius 2 (the green circles in the pictures below).*

1. 평범한 GAN 을 시드 2 로 10,000 걸음 학습하면, 군집 8개를 모두 덮을까요, 일부만 덮을까요? 원하면 수를 찍어 보세요. 시드만 바꾸면 결과는 같을까요?
   *Train a plain GAN with seed 2 for 10,000 steps. Will it cover all 8 clusters, or only some of them? Guess a number if you like. Would the result stay the same with another seed?*
2. 군집이 빠진다면, 손실 곡선(d_loss, g_loss)만 보고 그것을 알 수 있을까요?
   *If clusters go missing, can you tell from the loss curves (d_loss, g_loss) alone?*
3. §3 에서 같은 고리, 같은 시작 시드로 WGAN-GP 를 돌립니다. 덮는 군집 수가 더 안정적일까요, 덜 안정적일까요, 비슷할까요? 왜 그렇게 생각하나요?
   *In §3 you run WGAN-GP on the same ring with the same starting seed. Do you expect coverage to be steadier, less steady, or similar? Why?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your predictions here. This cell does not need to be run.
my_prediction_1 = ""
my_prediction_2 = ""
my_prediction_3 = ""

### §1 GAN 을 돌려 군집 세기 / Run a GAN and count the clusters
`train_gan` 은 두 단계를 한 걸음으로 묶어 되풀이합니다. 단계 A 는 G 의 가중치를 그대로 둔 채 D 를 갱신하고, 단계 B 는 D 를 지나는 기울기로 G 를 갱신합니다(이때 D 의 가중치는 갱신되지 않지만 기울기는 D 를 지나갑니다. 슬라이드 「What Loss Do We Actually Use for G?」의 「Backward」 줄).
*`train_gan` repeats two steps as one training step. Step A updates D while G's weights stay put; step B updates G through D, and D's weights do not update even though the gradient passes through D (the "Backward" line of the slide "What Loss Do We Actually Use for G?").*

두 단계의 손실 한 줄씩은 **§2 에서 여러분이 직접 씁니다.** 그래서 §1 은 그 두 줄을 아래 접힌 셀(▶)에 넣어 두었습니다. 지금은 열지 말고 실행만 하세요.
*You write the one-line loss of each step **yourself in §2**. For §1, those two lines sit in the folded cell (▶) below. Run it without opening it.*

**고정 z** 는 확인할 때마다 다시 쓰는 같은 잡음 벡터입니다(슬라이드 「New Problem ②  G and D Keep Chasing Each Other」의 「Fixed z」).
*Fixed z: the same noise vectors, reused at every check.*

「덮었다」의 기준. 고정 z 1,000개로 만든 점 가운데 50개(5%) 이상이 군집 중심에서 0.2(표준편차 4배) 안에 떨어지면 그 군집을 덮었다고 셉니다.
*"Covered" means: of the 1,000 points made from the fixed z, at least 50 (5%) land within 0.2 (4 standard deviations) of that cluster's center.*

이 기준은 이 실습을 위한 작업 정의이지, GAN 의 표준 지표가 아닙니다.
*This is our working definition for this lab, not a standard GAN metric.*

「군집 밖 비율」도 함께 셉니다. 1,000개 가운데 어느 군집 중심에서도 0.2 안에 들지 못한 점의 비율입니다. 덮은 군집이 8개여도 점이 군집 사이에 흩어져 있으면 이 비율이 큽니다.
*We also count the "off-cluster" share: of the 1,000 points, the share not within 0.2 of any center. Even with all 8 clusters covered, it is large when points are scattered between them.*

In [ ]:
#@title ▶ §1 에서 쓰는 손실 두 줄. §2 에서 직접 씁니다. 실행만 하세요 / The two loss lines used in §1. You write them in §2; just run this { display-mode: "form" }
def ref_d_loss(D, x_real, x_fake):
    return bce(D(x_real), 1) + bce(D(x_fake.detach()), 0)

def ref_g_loss(D, x_fake):
    return bce(D(x_fake), 1)

In [ ]:
def train_gan(d_loss_fn, g_loss_fn, seed=2, steps=10000, snap_steps=()):
    """평범한 GAN 학습. 한 걸음 = D 한 번 갱신(단계 A), 그다음 G 한 번 갱신(단계 B).
    Plain GAN training. One step = one D update (step A), then one G update (step B)."""
    torch.manual_seed(seed)                     # 같은 시드 = 같은 처음 가중치, 같은 데이터 순서 / same seed, same initial weights and data order
    G, D = make_G(), make_D()
    opt_G = torch.optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.999))   # G 만 움직이는 옵티마이저 / moves only G
    opt_D = torch.optim.Adam(D.parameters(), lr=1e-3, betas=(0.5, 0.999))   # D 만 움직이는 옵티마이저 / moves only D
    snaps, hist = {}, []
    for step in range(1, steps + 1):
        x_real = sample_ring(B)                 # 진짜 점 256개 / 256 real points
        z = torch.randn(B, Z_DIM)               # 잡음 256개 / 256 noise vectors
        x_fake = G(z)                           # 생성한 점 256개 / 256 generated points
        d_loss = d_loss_fn(D, x_real, x_fake)   # 단계 A. D 의 손실 / step A, D's loss
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()   # D 만 한 걸음 / one step for D only
        g_loss = g_loss_fn(D, x_fake)           # 단계 B. 같은 x_fake 로 G 의 손실 / step B, G's loss on the same x_fake
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()   # G 만 한 걸음 / one step for G only
        if step % 100 == 0:
            hist.append((step, d_loss.item(), g_loss.item()))   # 100 걸음마다 손실 기록 / record the losses every 100 steps
        if step in snap_steps:
            snaps[step] = coverage(G)           # 고정 z 로 그림과 덮은 군집 수 / picture and covered count from the fixed z
    return G, snaps, hist

In [ ]:
t0 = time.time()
G1, snaps_gan, hist_gan = train_gan(ref_d_loss, ref_g_loss, seed=2, steps=10000, snap_steps=(3000, 5000, 6000, 10000))
print(f"걸린 시간 / time: {time.time() - t0:.0f} s")
for s, (n_cov, counts, _) in snaps_gan.items():
    print(f"step {s:6,}: 덮은 군집 / covered {n_cov} of 8   군집 밖 / off-cluster {off_cluster(counts):.1%}   군집별 점 수 / points per cluster {counts}")
show_rings(snaps_gan, "GAN")
# 기대값 / Expected   이 노트북을 만든 2코어 CPU 에서 약 35초 / about 35 s on the 2-core CPU used to build this notebook
#   step 3,000: 2 of 8, 27.1%   step 5,000: 3 of 8, 10.7%   step 6,000: 3 of 8, 10.5%   step 10,000: 4 of 8, 5.3%
#   앞의 세 그림의 점은 슬라이드 「New Problem ①  G Covers Only Part of the Data」의 세 그림과 같습니다(제목 글자만 다름).
#   The points in the first three pictures are those on that slide (only the titles differ).
#   다른 컴퓨터에서는 숫자가 다를 수 있습니다. GAN 학습은 아주 작은 반올림 차이에도 다른 길로 갑니다.
#   같은 런타임에서 다시 실행하면 같은 숫자가 나옵니다.
#   On another computer the numbers can differ: GAN training takes another path after tiny rounding differences.
#   Re-running on the same runtime gives the same numbers.

In [ ]:
h = np.array(hist_gan)                        # 열: 걸음, d_loss, g_loss / columns: step, d_loss, g_loss
plt.figure(figsize=(6.5, 2.6))
plt.plot(h[:, 0], h[:, 1], color=GREEN, lw=1, label="d_loss")
plt.plot(h[:, 0], h[:, 2], color=GOLD, lw=1, label="g_loss")
plt.xlabel("step"); plt.ylabel("loss (nats)"); plt.title("GAN, seed 2: the two losses"); plt.legend(); plt.show()
for s in (3000, 6000, 10000):
    row = h[h[:, 0] == s][0]
    print(f"step {s:6,}: d_loss {row[1]:.3f}   g_loss {row[2]:.3f}")
# 기대값 / Expected
#   step  3,000: d_loss 0.808   g_loss 2.080
#   step  6,000: d_loss 0.947   g_loss 1.597
#   step 10,000: d_loss 0.958   g_loss 1.293
#   군집 4개가 빠졌지만, 두 곡선만 보고 그것을 알아내기는 어렵습니다.
#   Four clusters are missing, but that is hard to tell from the two curves alone.

**What to notice.** 시드 2 의 GAN 은 10,000 걸음 뒤 군집 8개 중 4개만 덮었습니다(이 노트북을 만든 환경). 5,000 걸음에서 6,000 걸음 사이에는 아래쪽 점 무리가 옆 군집으로 통째로 옮겨 갔습니다. G 는 자기가 만든 점에 대한 D 의 판정으로만 배웁니다. 그래서 한 번도 가지 않는 군집에서는 G 에게 직접 오는 신호가 없습니다. 이상적인 최적점 p_g = p_data 에서는 모든 군집을 덮으니, 문제는 거기까지 가는 학습 과정입니다. 손실값만으로는 군집이 빠졌는지 믿을 만하게 알 수 없습니다. 그래서 **표본을 보고, 덮은 군집 수와 군집 밖 비율을 셉니다**(슬라이드 「New Problem ①  G Covers Only Part of the Data」의 꼬리줄).
*With seed 2, the GAN covered only 4 of the 8 clusters after 10,000 steps (in the environment used to build this notebook). Between steps 5,000 and 6,000, the bottom group moved over to a neighboring cluster in one piece. G learns only from D's verdict on the points it makes, so a cluster it never visits sends it no direct signal. At the ideal optimum p_g = p_data every cluster is covered; the trouble is the path of training. Loss values alone cannot reliably tell you a cluster is missing, so **look at the samples and count the covered clusters and the off-cluster share**.*

> 요약: 손실값만으로는 모드 누락을 믿을 만하게 진단할 수 없다. 표본을 보고, 덮은 군집 수와 군집 밖 비율을 센다.

## §2 손실 두 줄을 직접 쓰기 / Write the two loss lines yourself
§1 의 두 단계를 이번에는 손실 한 줄씩으로 직접 씁니다. 「Backward」 줄은 슬라이드 「What Loss Do We Actually Use for G?」, `.detach()` 를 빠뜨리는 실수는 「Four Ways to Get This Wrong」 1번입니다.
*Now write the two steps of §1 as one loss line each. The "Backward" line is on the slide "What Loss Do We Actually Use for G?"; forgetting `.detach()` is item 1 of "Four Ways to Get This Wrong".*

```
x_fake = G(z)
단계 A / step A   d_loss : D 를 학습. 진짜는 1, 생성은 0 으로 맞히게. G 는 움직이면 안 된다.
                           trains D to say 1 for real and 0 for generated. G must not move.
단계 B / step B   g_loss : G 를 학습. D 가 생성 점을 진짜(1)라고 말하게. 기울기가 D 를 지나 G 로 간다.
                           trains G to make D say 1 (real) for generated points. The gradient passes through D into G.
```

- 이 노트북의 `D(x)` 는 시그모이드 앞의 점수 a(로짓)를 냅니다. `bce` 가 안에서 시그모이드를 붙여 p = σ(a) 로 바꿉니다(슬라이드 「From D's Raw Score to a Probability」). 그래서 −log 0 같은 수치 문제도 생기지 않습니다.
  *In this notebook `D(x)` returns the raw score a (a logit) before the sigmoid; `bce` applies the sigmoid inside, turning it into p = σ(a). This also avoids numerical trouble such as −log 0.*
- `bce(D(x), 1)` 은 「x 를 진짜(1)로 맞혀라」, `bce(D(x), 0)` 은 「x 를 생성(0)으로 맞혀라」입니다(슬라이드 「D's Loss Is Binary Cross-Entropy」).
  *`bce(D(x), 1)` says "call x real (1)"; `bce(D(x), 0)` says "call x generated (0)".*
- G 의 손실은 **비포화형**입니다(「What Loss Do We Actually Use for G?」, 「Optional D · Why the First Version Stalls Early」). minimax 식의 log(1 − D) 를 줄이는 대신, D 가 생성 점에 1 을 말하도록 정답 1 로 BCE 를 씁니다.
  *G's loss is the **non-saturating** one: instead of minimizing log(1 − D) from the minimax formula, use BCE with target 1, so that D says 1 for generated points.*
- `.detach()` 는 값은 그대로 두고, 그 앞으로 기울기를 보내지 않습니다. 5주차 VQ-VAE 의 stop-gradient 와 같은 도구입니다.
  *`.detach()` keeps the value but sends no gradient back past it, the same tool as Week 5's stop-gradient in the VQ-VAE.*

`train_gan` 은 단계 A 의 `x_fake` 를 단계 B 에서 **그대로 다시 씁니다**. 이 점이 TODO 1 에서 중요합니다.
*`train_gan` **reuses** step A's `x_fake` in step B. This matters for TODO 1.*

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   D       : 판별자. D(x) 는 점마다 로짓 하나, 크기 (B, 1) / discriminator; D(x) gives one logit per point, shape (B, 1)
#   x_real  : 진짜 점들. 크기 (B, 2) / real points, shape (B, 2)
#   x_fake  : G 가 만든 점들. G 까지 이어진 계산 그래프를 달고 있다. 크기 (B, 2)
#             points made by G; they carry the computation graph back to G, shape (B, 2)
#   bce(logits, target) : §0 의 이진 교차엔트로피. target 은 1(진짜) 또는 0(생성)
#                         binary cross-entropy from §0; target is 1 (real) or 0 (generated)
#   d_loss  : D 의 손실, 숫자 하나   <- 여기가 빈칸 / D's loss, a single number   <- this is the blank
#
# 참고 / Reference   t.detach()
#   t 와 같은 값을 갖되, 기울기 계산에서는 상수로 취급되는 텐서를 돌려준다. 그 앞으로는 기울기가 가지 않는다.
#   Returns a tensor with the same value as t, treated as a constant for gradients. No gradient flows back past it.
#   예 / e.g.   a = torch.tensor(3., requires_grad=True);  (2 * a.detach() + a).backward();  a.grad  ->  1.0
#
# 참고 / Reference   bce(logits, target)
#   예 / e.g.   bce(torch.tensor([[0.0]]), 1)  ->  0.6931  (= ln 2. 로짓 0 은 확률 ½ / a logit of 0 is a probability of ½)

def d_loss_fn(D, x_real, x_fake):
    """단계 A. 진짜는 1, 생성은 0 으로 맞히도록 D 를 학습시키는 손실.
    Step A. The loss that trains D to say 1 for real points and 0 for generated points."""
    # TODO. 진짜 점에는 정답 1, 생성 점에는 정답 0 을 주는 BCE 두 항을 더하세요.
    #       이 손실은 G 를 움직이면 안 됩니다. 생성 점에 무엇을 붙여야 할까요? 한 줄입니다.
    # TODO. Add two BCE terms: target 1 for the real points, target 0 for the generated points.
    #       This loss must not move G. What must you attach to the generated points? One line.
    d_loss = None

    if d_loss is None:
        raise NotImplementedError(BLANK_MSG)

    return d_loss

# 자가검증 / Check your work
# 작은 G 하나와, 점의 첫 좌표를 그대로 로짓으로 내는 D 하나로 확인합니다.
# Check with a tiny G and a D whose logit is simply the first coordinate of the point.
G_chk = nn.Linear(1, 2)                                     # z 하나를 점 하나로 / one z to one point
D_chk = nn.Linear(2, 1, bias=False)                         # 로짓 = 첫 좌표 / logit = first coordinate
with torch.no_grad():
    G_chk.weight.zero_(); G_chk.bias.copy_(torch.tensor([-1.0, 0.0]))   # 어떤 z 든 점 (-1, 0) / always the point (-1, 0)
    D_chk.weight.copy_(torch.tensor([[1.0, 0.0]]))
x_real_chk = torch.tensor([[2.0, 0.0]])                     # 진짜 점. 로짓 2 / a real point, logit 2
x_fake_chk = G_chk(torch.ones(1, 1))                        # 생성 점. 로짓 -1, G 까지 그래프가 이어져 있다 / a generated point, logit -1, graph back to G

loss_chk = d_loss_fn(D_chk, x_real_chk, x_fake_chk)
loss_chk.backward()
print(f"d_loss = {loss_chk.item():.4f}")
# 기대값 / Expected   0.4402  ( = ln(1 + e^-2) + ln(1 + e^-1) = 0.1269 + 0.3133 )
#   3.4402 가 나오면 정답 1 과 0 을 바꿔 쓴 것입니다.
#   3.4402 means the targets 1 and 0 were swapped.
#   0.1269 나 0.3133 이 나오면 두 항 중 하나가 빠졌습니다.
#   0.1269 or 0.3133 means one of the two terms is missing.
print("G 에 온 기울기 / gradient that reached G:", G_chk.weight.grad)
# 기대값 / Expected   None
#   None 대신 숫자가 찍히면 .detach() 가 빠져 D 의 손실이 G 까지 기울기를 보낸 것입니다.
#   Numbers instead of None mean .detach() is missing, so D's loss sent gradients into G.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요. 이미 채웠다면 실행하지 마세요(내 함수를 덮어씁니다) / Only if stuck. If you already filled in the blank, skip it: it overwrites your function { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

def d_loss_fn(D, x_real, x_fake):
    """단계 A. 진짜는 1, 생성은 0 으로 맞히도록 D 를 학습시키는 손실.
    Step A. The loss that trains D to say 1 for real points and 0 for generated points."""
    d_loss = bce(D(x_real), 1) + bce(D(x_fake.detach()), 0)   # 빈칸이었던 줄. 생성 점은 .detach() 로 G 와 끊는다
                                                              # the blank. .detach() cuts the generated points off from G

    if d_loss is None:
        raise NotImplementedError(BLANK_MSG)

    return d_loss

# 자가검증 / Check your work
# 작은 G 하나와, 점의 첫 좌표를 그대로 로짓으로 내는 D 하나로 확인합니다.
# Check with a tiny G and a D whose logit is simply the first coordinate of the point.
G_chk = nn.Linear(1, 2)                                     # z 하나를 점 하나로 / one z to one point
D_chk = nn.Linear(2, 1, bias=False)                         # 로짓 = 첫 좌표 / logit = first coordinate
with torch.no_grad():
    G_chk.weight.zero_(); G_chk.bias.copy_(torch.tensor([-1.0, 0.0]))   # 어떤 z 든 점 (-1, 0) / always the point (-1, 0)
    D_chk.weight.copy_(torch.tensor([[1.0, 0.0]]))
x_real_chk = torch.tensor([[2.0, 0.0]])                     # 진짜 점. 로짓 2 / a real point, logit 2
x_fake_chk = G_chk(torch.ones(1, 1))                        # 생성 점. 로짓 -1, G 까지 그래프가 이어져 있다 / a generated point, logit -1, graph back to G

loss_chk = d_loss_fn(D_chk, x_real_chk, x_fake_chk)
loss_chk.backward()
print(f"d_loss = {loss_chk.item():.4f}")
# 기대값 / Expected   0.4402  ( = ln(1 + e^-2) + ln(1 + e^-1) = 0.1269 + 0.3133 )
#   3.4402 가 나오면 정답 1 과 0 을 바꿔 쓴 것입니다.
#   3.4402 means the targets 1 and 0 were swapped.
#   0.1269 나 0.3133 이 나오면 두 항 중 하나가 빠졌습니다.
#   0.1269 or 0.3133 means one of the two terms is missing.
print("G 에 온 기울기 / gradient that reached G:", G_chk.weight.grad)
# 기대값 / Expected   None
#   None 대신 숫자가 찍히면 .detach() 가 빠져 D 의 손실이 G 까지 기울기를 보낸 것입니다.
#   Numbers instead of None mean .detach() is missing, so D's loss sent gradients into G.

In [ ]:
# 다음 이름을 씁니다 / Names used below
#   D       : 판별자 / discriminator
#   x_fake  : G 가 만든 점들. 이번에는 기울기가 D 를 지나 G 까지 가야 한다
#             points made by G; this time the gradient must pass through D into G
#   g_loss  : G 의 손실, 숫자 하나   <- 여기가 빈칸 / G's loss, a single number   <- this is the blank
#
# 참고 / Reference   bce(logits, target), §0
#   예 / e.g.   bce(torch.tensor([[2.0]]), 0)  ->  2.1269  (= ln(1 + e^2))

def g_loss_fn(D, x_fake):
    """단계 B. D 가 생성 점을 진짜(1)라고 말하도록 G 를 학습시키는 손실. 비포화형.
    Step B. The non-saturating loss that trains G to make D say 1 (real) for generated points."""
    # TODO. G 는 D 가 생성 점을 무엇이라고 말하기를 원하나요? 그 정답으로 BCE 한 항을 쓰세요.
    #       이번에는 .detach() 를 붙이지 않습니다. 기울기가 D 를 지나 G 까지 가야 하니까요. 한 줄입니다.
    # TODO. What does G want D to say about the generated points? Write one BCE term with that target.
    #       No .detach() this time: the gradient must pass through D into G. One line.
    g_loss = None

    if g_loss is None:
        raise NotImplementedError(BLANK_MSG)

    return g_loss

# 자가검증 / Check your work. 위 TODO 1 셀의 G_chk, D_chk 를 다시 씁니다 / reuses G_chk and D_chk from the TODO 1 cell
G_chk.zero_grad(set_to_none=True)                           # G 의 기울기를 비운다 / clear G's gradients
x_fake_chk = G_chk(torch.ones(1, 1))                        # 생성 점을 다시 만든다. 로짓 -1 / make the generated point again, logit -1
loss_g = g_loss_fn(D_chk, x_fake_chk)
loss_g.backward()
print(f"g_loss = {loss_g.item():.4f}")
# 기대값 / Expected   1.3133  ( = -ln D = ln(1 + e^1), D = sigmoid(-1) = 0.2689 )
#   -0.3133 이 나오면 포화형, 곧 minimax 식의 ln(1 - D) 를 그대로 줄이는 식을 쓴 것입니다.
#   -0.3133 means the saturating form, minimizing ln(1 - D) from the minimax formula as written.
#   0.3133 이 나오면 정답을 0 으로 쓴 것입니다. 그러면 G 가 D 를 돕는 쪽으로 움직입니다.
#   0.3133 means the target is 0, so G would move to help D.
print("G 에 온 기울기 / gradient that reached G:", G_chk.weight.grad)
# 기대값 / Expected   숫자가 든 텐서. None 이 아님 / a tensor of numbers, not None
#   None 이면 g_loss 에 .detach() 를 붙여 G 로 가는 길을 끊은 것입니다.
#   None means a .detach() in g_loss cut the path to G.

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요. 이미 채웠다면 실행하지 마세요(내 함수를 덮어씁니다) / Only if stuck. If you already filled in the blank, skip it: it overwrites your function { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.

def g_loss_fn(D, x_fake):
    """단계 B. D 가 생성 점을 진짜(1)라고 말하도록 G 를 학습시키는 손실. 비포화형.
    Step B. The non-saturating loss that trains G to make D say 1 (real) for generated points."""
    g_loss = bce(D(x_fake), 1)   # 빈칸이었던 줄. 정답 1, .detach() 없음
                                 # the blank. Target 1, no .detach()

    if g_loss is None:
        raise NotImplementedError(BLANK_MSG)

    return g_loss

# 자가검증 / Check your work. 위 TODO 1 셀의 G_chk, D_chk 를 다시 씁니다 / reuses G_chk and D_chk from the TODO 1 cell
G_chk.zero_grad(set_to_none=True)                           # G 의 기울기를 비운다 / clear G's gradients
x_fake_chk = G_chk(torch.ones(1, 1))                        # 생성 점을 다시 만든다. 로짓 -1 / make the generated point again, logit -1
loss_g = g_loss_fn(D_chk, x_fake_chk)
loss_g.backward()
print(f"g_loss = {loss_g.item():.4f}")
# 기대값 / Expected   1.3133  ( = -ln D = ln(1 + e^1), D = sigmoid(-1) = 0.2689 )
#   -0.3133 이 나오면 포화형, 곧 minimax 식의 ln(1 - D) 를 그대로 줄이는 식을 쓴 것입니다.
#   -0.3133 means the saturating form, minimizing ln(1 - D) from the minimax formula as written.
#   0.3133 이 나오면 정답을 0 으로 쓴 것입니다. 그러면 G 가 D 를 돕는 쪽으로 움직입니다.
#   0.3133 means the target is 0, so G would move to help D.
print("G 에 온 기울기 / gradient that reached G:", G_chk.weight.grad)
# 기대값 / Expected   숫자가 든 텐서. None 이 아님 / a tensor of numbers, not None
#   None 이면 g_loss 에 .detach() 를 붙여 G 로 가는 길을 끊은 것입니다.
#   None means a .detach() in g_loss cut the path to G.

### 내 두 줄로 다시 돌리기 / Run again with your two lines
§1 과 같은 학습을 여러분의 `d_loss_fn`, `g_loss_fn` 으로 다시 돌립니다. 정답과 같은 식으로 썼다면 §1 과 숫자가 그대로 같습니다.
*The same training as §1, now with your `d_loss_fn` and `g_loss_fn`. If you wrote them as in the solution, the numbers match §1 exactly.*

In [ ]:
t0 = time.time()
G2, snaps_mine, hist_mine = train_gan(d_loss_fn, g_loss_fn, seed=2, steps=10000, snap_steps=(10000,))
print(f"걸린 시간 / time: {time.time() - t0:.0f} s")
n_mine, counts_mine, _ = snaps_mine[10000]
n_ref, counts_ref, _ = snaps_gan[10000]
print("내 두 줄 / my two lines     :", n_mine, "of 8", counts_mine)
print("§1 의 두 줄 / §1's two lines :", n_ref, "of 8", counts_ref)
print("같은가 / identical:", counts_mine == counts_ref)
# 기대값 / Expected   identical: True
#   위 두 자가검증을 통과했는데 False 라면, 같은 식을 조금 다르게 쓴 것입니다(예: 시그모이드를 따로 씌운 식).
#   그런 작은 차이도 GAN 학습을 다른 길로 보냅니다. 틀린 것은 아닙니다.
#   False after passing both checks means the same formula written slightly differently (for example, a separate sigmoid).
#   Even that sends GAN training down another path. It is not wrong.
#   "Trying to backward through the graph a second time" 오류가 나면 d_loss 에 .detach() 가 빠진 것입니다.
#   단계 A 의 역전파가 G 까지 그래프를 써 버려서, 단계 B 가 같은 x_fake 로 다시 역전파할 수 없습니다.
#   The error "Trying to backward through the graph a second time" means .detach() is missing in d_loss:
#   step A's backward pass used up the graph back to G, so step B cannot backpropagate through the same x_fake again.

**What to notice.** `.detach()` 하나가 두 가지를 막습니다. D 의 손실이 G 의 가중치에 기울기를 보내는 것, 그리고 단계 A 가 G 쪽 계산 그래프를 써 버려 단계 B 의 역전파가 실패하는 것입니다(슬라이드 「Four Ways to Get This Wrong」 1번). 기울기를 비우는 순서에 따라서는 오류 대신 G 가 D 를 돕는 쪽으로 조용히 움직이기도 합니다. G 의 손실에는 반대로 `.detach()` 가 없어야 기울기가 D 를 지나 G 에 닿습니다.
*One `.detach()` prevents two things: D's loss sending gradients into G's weights, and step A using up the graph on G's side so that step B's backward pass fails (pitfall 1 on the slide). Depending on when the gradients are cleared, the silent version also happens: G drifts toward helping D. G's loss is the opposite: no `.detach()`, so the gradient passes through D and reaches G.*

> 요약: D 의 손실에는 `.detach()`, G 의 손실에는 없음.

## §3 WGAN-GP 로 바꾸기. 같은 고리, 같은 시작 시드 / Switch to WGAN-GP. Same ring, same starting seed
**왜 바꾸나.** 생성 점이 진짜에서 완전히 떨어져 있으면, 이상적인 D 는 가까운 쪽이든 먼 쪽이든 똑같이 「generated」라고 합니다(슬라이드 「Closer, but D Still Says “Generated”」). WGAN-GP 는 「얼마나 남았나」에 따라 바뀌는 점수를 씁니다(「A Score That Changes With Every Step Closer」).
*Why switch. When generated points sit completely apart from the real ones, an ideal D says "generated" to a closer and a farther set alike (slide "Closer, but D Still Says “Generated”"). WGAN-GP uses a score that changes with how far there is still to go ("A Score That Changes With Every Step Closer").*

**바뀌는 것 셋.** 네트워크 모양, 학습률 1e-3, 데이터, 시작 시드는 §1 과 같습니다.
*Three things change. The network shapes, the learning rate 1e-3, the data and the starting seed are the same as in §1.*

1. D 가 비평가(critic)가 됩니다. 출력은 확률이 아니라 점수 f(x), 아무 실수입니다.
   *D becomes a critic: its output is a score f(x), any real number, not a probability.*
2. 비평가는 진짜 점의 평균 점수와 생성 점의 평균 점수를 비교합니다. G 는 생성 점의 평균 점수를 올립니다.
   *The critic compares the average real score with the average generated score. G raises the average generated score.*
3. 기울기 벌점이 비평가의 점수가 너무 급하게 바뀌지 못하게 막습니다. 그래야 점수 차가 거리(Wasserstein 거리 W)의 추정이 됩니다.
   *A gradient penalty keeps the critic from changing its score too sharply, so that the score gap estimates a distance (the Wasserstein distance W).*

**오늘의 실습 목표는 WGAN-GP 식을 유도하는 것이 아닙니다.** 학습 신호를 바꾸었을 때 방금 §1 에서 본 군집 덮기와 학습 행동이 어떻게 달라지는지를 관찰하는 것입니다.
*Today's question is not "can you derive WGAN-GP?" It is "does a different training signal change the mode coverage and training behavior we just observed in §1?"*

아래 식은 주어진 것입니다. 오늘 유도하지 않습니다. 코드도 이미 다 적혀 있으니 실행만 하세요.
*The formula below is provided; you do not need to derive it today. The code is already written; just run it.*

$$\text{critic loss} = \mathbb{E}[f(G(z))] - \mathbb{E}[f(x)] + \lambda\,\mathbb{E}_{\hat x}\big[(\lVert\nabla f(\hat x)\rVert - 1)^2\big],\qquad \hat x = \varepsilon x + (1-\varepsilon)\,G(z),\ \ \varepsilon \sim U[0, 1]$$

| | GAN (§1, §2) | WGAN-GP (§3) |
|---|---|---|
| D 의 출력 / D's output | 로짓, 시그모이드로 확률 / a logit, a probability after the sigmoid | 점수 f(x), 아무 실수 / a score f(x), any real number |
| D 의 손실 / D's loss | BCE 두 항 / two BCE terms | −(진짜 평균 점수 − 생성 평균 점수) + λ · 기울기 벌점<br>−(average real score − average generated score) + λ · gradient penalty |
| G 의 손실 / G's loss | `bce(D(x_fake), 1)` | −(생성 점의 평균 점수) / −(average score of generated points) |
| 갱신 비율 / updates | D 1번 : G 1번 / D 1 : G 1 | 비평가 5번 : G 1번 / critic 5 : G 1 |

**실행 전에 알아 둘 것. 이것은 한 가지만 바꾼 통제 실험이 아닙니다.** WGAN-GP 는 손실만이 아니라 비평가 갱신 비율과 목적도 바꿉니다. 그래서 아래 결과는 「두 방법의 학습 행동 비교」이지, 「어느 한 변화 때문」을 증명하는 것이 아닙니다. 시작 시드가 같아도 비평가가 5번씩 돌면 그 뒤 데이터와 z 를 뽑는 순서는 GAN 과 달라집니다.
*Before you run: this is not a controlled ablation of one ingredient. WGAN-GP also changes the critic update ratio and the training objective. So the results below compare training behavior; they do not prove that one change caused the result. Even with the same starting seed, the critic's 5 steps per G step change the later order of data and z draws.*

| | GAN (§1) | WGAN-GP (§3) |
|---|---|---|
| G 갱신 횟수 / G updates | 10,000 | 3,000 |
| D · 비평가 갱신 횟수 / D or critic updates | 10,000 | 15,000 (G 한 번에 5번 / 5 per G update) |
| Adam (β1, β2) | (0.5, 0.999) | (0.5, 0.9) |
| D 한 걸음의 계산 / cost of one D step | BCE 두 항 / two BCE terms | 점수 차 + 기울기 벌점. 기울기의 기울기라 더 무거움 / score gap + gradient penalty, a gradient of a gradient, so heavier |
| 시드가 맞춰 주는 것 / what the seed fixes | 처음 가중치 / initial weights | 같은 처음 가중치. 그 뒤 데이터와 z 를 뽑는 순서는 다름 / the same initial weights; after that, data and z are drawn in a different order |

<details><summary>더 알고 싶다면 / For the curious</summary>

- 비평가가 키우는 점수 차는 기울기가 제한될 때 Wasserstein 거리 W 의 추정이 됩니다(슬라이드 「D Becomes a Critic」, 「Optional G · Why the Critic's Slope Must Be Limited」). 기울기 벌점은 진짜 점과 생성 점을 잇는 직선 위의 무작위 점 x̂ 에서, 비평가의 기울기 크기가 1 에서 벗어난 만큼을 제곱해 더합니다(「Optional H · The Penalty That Keeps Slopes Near 1」).
  *The score gap the critic enlarges becomes an estimate of the Wasserstein distance W when its slope is limited. The gradient penalty adds the squared distance of the critic's slope from 1, measured at a random point x̂ on the line between a real and a generated point.*
- λ = 10, 비평가 5걸음은 WGAN-GP 논문(Gulrajani 외, NIPS 2017)의 기본값입니다. 논문의 Adam 은 β1 = 0, β2 = 0.9 이고, 여기서는 β1 만 §1 과 같은 0.5 로 두었습니다.
  *λ = 10 and 5 critic steps are the paper's defaults. The paper's Adam uses β1 = 0, β2 = 0.9; here only β1 is kept at §1's 0.5.*
- 비평가에는 **배치 정규화가 없습니다.** 벌점은 점 하나하나에 대한 기울기인데, 배치 정규화는 배치 안의 점들을 서로 섞기 때문입니다(논문 §4). 과제 2 에서 DCGAN 구조를 쓸 때 기억하세요.
  *The critic has **no batch normalization**: the penalty is a per-point gradient, and batch normalization mixes the points of a batch (paper §4). Remember this when you use a DCGAN architecture in Assignment 2.*

</details>


In [ ]:
LAMBDA, N_CRITIC = 10.0, 5      # 벌점 무게 λ, G 한 걸음당 비평가 걸음 수. WGAN-GP 논문의 기본값 / penalty weight λ, critic steps per G step: the paper's defaults

def critic_loss(f, x_real, x_fake):
    """비평가 f 의 손실 = -(점수 차) + λ · 기울기 벌점. 점수 차(W 의 추정)도 함께 돌려준다.
    Critic loss = -(score gap) + λ * gradient penalty. Also returns the score gap, the estimate of W."""
    x_fake = x_fake.detach()                                    # 비평가 단계에서는 G 가 움직이지 않는다(§2 와 같은 이유) / G must not move in the critic step (same reason as §2)
    eps = torch.rand(x_real.size(0), 1)                         # 점마다 ε ~ U[0, 1] / one ε per point
    x_hat = (eps * x_real + (1 - eps) * x_fake).requires_grad_(True)   # 진짜와 생성을 잇는 직선 위 무작위 점 x̂ / a random point on the line between real and generated
    grad = torch.autograd.grad(f(x_hat).sum(), x_hat, create_graph=True)[0]
    # 위 줄: x̂ 에서의 기울기 ∇f(x̂). create_graph=True 는 이 기울기 계산까지 다시 미분할 수 있게 한다(기울기의 기울기).
    # Line above: the slope ∇f(x̂). create_graph=True lets this gradient itself be differentiated (a gradient of a gradient).
    gp = ((grad.norm(dim=1) - 1) ** 2).mean()                   # 기울기 크기가 1 에서 벗어난 만큼의 제곱, 평균 / squared distance of the slope size from 1, averaged
    gap = f(x_real).mean() - f(x_fake).mean()                   # 진짜 평균 점수 - 생성 평균 점수 / average real score minus average generated score
    return -gap + LAMBDA * gp, gap.item()

def g_loss_wgan(f, x_fake):
    """G 의 손실 = -(생성 점의 평균 점수). G 는 점수 차를 줄인다 / G's loss = -(average score of generated points); G shrinks the gap"""
    return -f(x_fake).mean()

def train_wgan_gp(seed=2, g_steps=3000, snap_steps=()):
    """WGAN-GP 학습. G 한 걸음마다 비평가 다섯 걸음 / WGAN-GP training: five critic steps for every G step"""
    torch.manual_seed(seed)
    G, f = make_G(), make_D()          # 비평가 f 는 D 와 같은 모양. 마지막 시그모이드 없음, 배치 정규화 없음 / critic f has D's shape: no sigmoid at the end, no batch normalization
    opt_G = torch.optim.Adam(G.parameters(), lr=1e-3, betas=(0.5, 0.9))
    opt_f = torch.optim.Adam(f.parameters(), lr=1e-3, betas=(0.5, 0.9))
    snaps, hist = {}, []
    for step in range(1, g_steps + 1):
        for _ in range(N_CRITIC):                               # 비평가 다섯 걸음 / five critic steps
            x_real = sample_ring(B)
            z = torch.randn(B, Z_DIM)
            x_fake = G(z)
            loss_f, gap = critic_loss(f, x_real, x_fake)
            opt_f.zero_grad(); loss_f.backward(); opt_f.step()
        z = torch.randn(B, Z_DIM)                               # 그다음 G 한 걸음 / then one G step
        g_loss = g_loss_wgan(f, G(z))
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()
        if step % 20 == 0:
            hist.append((step, gap))                            # 마지막 비평가 배치의 점수 차 / score gap of the last critic batch
        if step in snap_steps:
            snaps[step] = coverage(G)
    return G, snaps, hist

In [ ]:
t0 = time.time()
G3, snaps_wgan, hist_wgan = train_wgan_gp(seed=2, g_steps=3000, snap_steps=(600, 1000, 3000))
print(f"걸린 시간 / time: {time.time() - t0:.0f} s")
for s, (n_cov, counts, _) in snaps_wgan.items():
    print(f"G step {s:5,} (비평가 / critic {N_CRITIC * s:6,}): 덮은 군집 / covered {n_cov} of 8   군집 밖 / off-cluster {off_cluster(counts):.1%}   {counts}")
show_rings(snaps_wgan, "WGAN-GP")
# 기대값 / Expected   약 55초 / about 55 s
#   G step 600: 5 of 8, 56.6%   G step 1,000: 8 of 8, 28.0%   G step 3,000: 8 of 8, 11.0%
#   다른 컴퓨터에서는 숫자가 다를 수 있습니다. 같은 런타임에서 다시 실행하면 같은 숫자가 나옵니다.
#   On another computer the numbers can differ. Re-running on the same runtime gives the same numbers.

In [ ]:
hg, hw = np.array(hist_gan), np.array(hist_wgan)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8))
axes[0].plot(hg[:, 0], hg[:, 1], color=GREEN, lw=1, label="d_loss")
axes[0].plot(hg[:, 0], hg[:, 2], color=GOLD, lw=1, label="g_loss")
axes[0].set_title("GAN, seed 2 (section 1): the two losses", fontsize=10); axes[0].set_xlabel("step"); axes[0].legend()
k = 10                                                          # 기록 10개 이동평균 / moving average over 10 records
smooth = np.convolve(hw[:, 1], np.ones(k) / k, mode="valid")
axes[1].plot(hw[:, 0], hw[:, 1], color=GOLD, lw=0.6, alpha=0.6, label="score gap")
axes[1].plot(hw[k - 1:, 0], smooth, color=GREEN, lw=1.5, label="moving average")
axes[1].set_title("WGAN-GP, seed 2: the critic's score gap (estimate of W)", fontsize=10); axes[1].set_xlabel("G step"); axes[1].legend()
plt.tight_layout(); plt.show()
print(f"점수 차 / score gap   G step {int(hw[0, 0])}: {hw[0, 1]:.3f}   G step {int(hw[-1, 0])}: {hw[-1, 1]:.3f}")
print("점수 차가 음수였던 기록 / records with a negative gap:", int((hw[:, 1] < 0).sum()), "of", len(hw))
# 기대값 / Expected
#   G step 20: 1.849   G step 3000: 0.061
#   점수 차가 음수였던 기록 / records with a negative gap: 1 of 150
#   점수 차는 생성 점이 군집 8개로 퍼져 가는 동안 줄어듭니다. 남은 거리 W 의 추정이기 때문입니다.
#   다만 배치 하나로 잰 추정이라 흔들리고, 가끔 음수도 찍힙니다. W 는 음수가 될 수 없으니 그 순간의 추정은 틀린 것입니다.
#   GAN 의 두 곡선에는 이런 내리막이 없습니다.
#   The gap falls while the generated points spread to all 8 clusters, because it estimates W, the distance still to go.
#   It is measured on one batch, so it wobbles and sometimes dips below zero. W cannot be negative, so such a reading is wrong.
#   The two GAN curves show no such descent.

**What to notice.** 같은 시드 2 에서 WGAN-GP 는 군집 8개를 모두 덮었고, 비평가의 점수 차는 그동안 줄어들었습니다. 점수 차는 W 의 추정이라 「얼마나 남았나」(슬라이드 「A Score That Changes With Every Step Closer」의 「Distance still to go」)를 대략 보여 주는 **진단 신호**입니다. GAN 의 손실 곡선보다 쓸모가 많지만 **보장은 아닙니다**(슬라이드 「Four Ways to Get This Wrong」 3번). WGAN-GP 논문 §5.6 도 점수 차가 표본 품질과 함께 움직였다고 보고하면서, 비평가가 과적합하면 W 를 잘못 추정하고 그때는 품질과의 관계도 믿을 수 없다고 적었습니다. 그리고 덮은 군집은 8개지만 군집 밖 비율은 11.0% 로 GAN(5.3%)보다 큽니다. 생성 점 일부가 군집 사이 원 위에 흩어져 남았습니다. 두 숫자를 같이 봐야 하는 이유입니다.
*With the same seed 2, WGAN-GP covered all 8 clusters, and the critic's score gap fell along the way. The gap estimates W, so it is a rough **diagnostic** of how far there is still to go. It says more than the GAN's loss curves, but it is **not a guarantee** (pitfall 3). Section 5.6 of the WGAN-GP paper reports that the gap moved with sample quality, and also that when the critic overfits it misestimates W and the link to quality can no longer be trusted. Also, 8 clusters are covered but 11.0% of the points are off-cluster, more than the GAN's 5.3%: some points stay spread along the circle between clusters. That is why both numbers matter.*

**보장은 아닙니다.** 이 노트북을 만든 환경에서 시드 0 ~ 4 를 모두 돌린 결과입니다.
*It is not a guarantee. Results for seeds 0 to 4 in the environment used to build this notebook:*

칸마다 덮은 군집 수, 괄호 안은 군집 밖 비율입니다.
*Each cell: clusters covered, with the off-cluster share in brackets.*

| 시드 / seed | GAN, 10,000 걸음 / steps | WGAN-GP, G 2,000 걸음 (비평가 10,000) / G steps (critic) | WGAN-GP, G 3,000 걸음 (비평가 15,000) / G steps (critic) |
|---|---|---|---|
| 0 | 8 (2.0%) | 8 (16.7%) | 8 (13.7%) |
| 1 | 7 (3.3%) | 5 (39.2%) | 8 (17.6%) |
| 2 | 4 (5.3%) | 8 (12.3%) | 8 (11.0%) |
| 3 | 6 (2.7%) | 0 (94.2%) | 7 (26.3%) |
| 4 | 6 (2.6%) | 8 (21.0%) | 8 (9.3%) |

시드 3 은 G 2,000 걸음에서 0개였습니다. 군집이 빠진 것이 아니라 점들이 아직 원 근처로 모이지 않고 넓게 흩어져 있었고(군집 밖 94.2%), 그동안 점수 차도 크게 흔들려 기록 150개 중 28개가 음수였습니다. 3,000 걸음에서는 7개를 덮었습니다. 「steadier GAN training」(슬라이드 「What the VAE, GAN and WGAN-GP Each Changed」)은 관찰이지 보장이 아닙니다. 흔들리는 점수 차는 「아직 안 끝났다」, 또는 「비평가가 W 를 잘 재지 못하고 있다」는 신호로 읽습니다.
*Seed 3 had 0 at 2,000 G steps. Not because clusters were dropped: the points had not yet gathered near the circle and were widely spread (94.2% off-cluster), and the score gap swung hard, with 28 of its 150 records below zero. By 3,000 steps it covered 7. "Steadier GAN training" (slide "What the VAE, GAN and WGAN-GP Each Changed") is an observation, not a guarantee. Read a swinging gap as "not done yet" or "the critic is not measuring W well".*

> 요약: 같은 시드에서 WGAN-GP 는 8개를 덮었고 점수 차가 줄었다. 점수 차는 진단 신호이지 보장이 아니고, 군집 밖 점은 더 많았다. 두 방법은 걸음 수와 설정도 달랐다.

**(선택) 다른 시드로 / (Optional) another seed.** 아래 셀의 `SEED_TRY` 를 0 ~ 4 중 하나로 바꿔 실행하면 두 방법을 같은 시드로 비교합니다(약 1분 30초).
*Set `SEED_TRY` in the cell below to one of 0 to 4 and run it to compare both methods on that seed (about 1.5 minutes).*

In [ ]:
SEED_TRY = 3      # 0 ~ 4 중 하나로 바꿔 보세요 / try one of 0 to 4
t0 = time.time()
_, s_gan, _ = train_gan(d_loss_fn, g_loss_fn, seed=SEED_TRY, steps=10000, snap_steps=(10000,))
_, s_wgan, _ = train_wgan_gp(seed=SEED_TRY, g_steps=3000, snap_steps=(2000, 3000))
print(f"걸린 시간 / time: {time.time() - t0:.0f} s")
print(f"시드 / seed {SEED_TRY}   GAN 10,000: {s_gan[10000][0]} of 8 ({off_cluster(s_gan[10000][1]):.1%})   "
      f"WGAN-GP 2,000: {s_wgan[2000][0]} of 8 ({off_cluster(s_wgan[2000][1]):.1%})   "
      f"WGAN-GP 3,000: {s_wgan[3000][0]} of 8 ({off_cluster(s_wgan[3000][1]):.1%})")
show_rings(s_gan, f"GAN seed {SEED_TRY}")
show_rings(s_wgan, f"WGAN-GP seed {SEED_TRY}")
# 기대값 / Expected   위 표의 그 시드 줄. 시드 3 이면 6 (2.7%), 0 (94.2%), 7 (26.3%) / the table row for that seed

## §4 (선택) MNIST 에서 작은 DCGAN / (Optional) A small DCGAN on MNIST
손실은 §2 에서 쓴 두 줄 그대로입니다. 바뀐 것은 네트워크뿐입니다. 합성곱 생성자(7 × 7 → 14 × 14 → 28 × 28)와 합성곱 판별자를 씁니다(슬라이드 「DCGAN Is a GAN Built for Images」의 DCGAN 생각을 28 × 28 MNIST 에 맞게 줄인 것. 「Did We Get Sharper Samples?」의 오른쪽 칸이 바로 이 작은 판이고, DCGAN 장에 그린 64 × 64 네트워크가 아닙니다). 판별자의 보폭 2 합성곱은 슬라이드 「Stride and Strided Convolution」, 생성자의 전치 합성곱은 「Transposed Convolution」입니다. 슬라이드 예는 커널 3 · 보폭 2 · 패딩 0(7 × 7 → 3 × 3, 3 × 3 → 7 × 7)이고, 여기서는 커널 4 · 보폭 2 · 패딩 1 이라 정확히 절반 · 두 배가 됩니다(28 → 14 → 7, 7 → 14 → 28). 전치 합성곱의 출력 크기는 (입력 − 1) × 보폭 − 2 × 패딩 + 커널 = (7 − 1) × 2 − 2 + 4 = 14 입니다.
*The losses are exactly your two lines from §2; only the networks change: a convolutional generator (7 × 7 → 14 × 14 → 28 × 28) and a convolutional discriminator, the DCGAN idea scaled down to 28 × 28 MNIST. The right panel of "Did We Get Sharper Samples?" is this small version, not the 64 × 64 network drawn on the DCGAN slide. D's stride-2 convolutions are the slide "Stride and Strided Convolution"; G's transposed convolutions are "Transposed Convolution". The slides use kernel 3, stride 2, padding 0 (7 × 7 → 3 × 3, 3 × 3 → 7 × 7); here kernel 4, stride 2, padding 1 halves or doubles exactly (28 → 14 → 7, 7 → 14 → 28). A transposed convolution's output size is (input − 1) × stride − 2 × padding + kernel = (7 − 1) × 2 − 2 + 4 = 14.*

CPU 에서 한 번 훑기(1에폭)에 약 75초 걸립니다. GPU 런타임이면 `EPOCHS = 5` 로 올려 보세요. 그림은 고정한 z 64개로 그립니다.
*One pass over the data (one epoch) takes about 75 seconds on a CPU. On a GPU runtime, try `EPOCHS = 5`. The grid uses 64 fixed z.*

In [ ]:
if mnist is None:                                         # §0 에서 내려받기에 실패했으면 다시 시도 / retry if the download in §0 failed
    mnist = datasets.MNIST("data", train=True, download=True, transform=transforms.ToTensor())
X = mnist.data.float().div(255).unsqueeze(1) * 2 - 1     # 이미지 60,000장. 값 [-1, 1], G 의 tanh 출력과 같은 범위 / 60,000 images in [-1, 1], the range of G's tanh output

class DCGen(nn.Module):
    """z(16) -> 128 x 7 x 7 -> 64 x 14 x 14 -> 1 x 28 x 28. 전치 합성곱이 가로세로를 두 배로 / transposed convolutions double height and width"""
    def __init__(self, zd=16):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(zd, 128 * 7 * 7), nn.BatchNorm1d(128 * 7 * 7), nn.ReLU())
        self.up = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),   # 7 -> 14
                                nn.ConvTranspose2d(64, 1, 4, 2, 1), nn.Tanh())                          # 14 -> 28
    def forward(self, z):
        return self.up(self.fc(z).view(-1, 128, 7, 7))

def make_dc_D():
    """보폭 2 합성곱 둘로 28 -> 14 -> 7, 그다음 로짓 하나. 이 D 에는 배치 정규화가 있다(DCGAN 요리법).
    Two stride-2 convolutions, 28 -> 14 -> 7, then one logit. This D has batch normalization (the DCGAN recipe)."""
    return nn.Sequential(nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),                           # 28 -> 14
                         nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),    # 14 -> 7
                         nn.Flatten(), nn.Linear(128 * 7 * 7, 1))

EPOCHS, BATCH = 1, 128
torch.manual_seed(0)
Gd, Dd = DCGen().to(device), make_dc_D().to(device)
opt_Gd = torch.optim.Adam(Gd.parameters(), lr=2e-4, betas=(0.5, 0.999))   # DCGAN 논문의 값 / the DCGAN paper's values
opt_Dd = torch.optim.Adam(Dd.parameters(), lr=2e-4, betas=(0.5, 0.999))
Z_GRID = torch.randn(64, 16, generator=torch.Generator().manual_seed(7)).to(device)   # 그림용 고정 z 64개 / 64 fixed z for the grid
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    perm = torch.randperm(len(X))                               # 매 에폭 순서를 섞는다 / shuffle every epoch
    for i in range(0, len(X), BATCH):
        x_real = X[perm[i:i + BATCH]].to(device)
        x_fake = Gd(torch.randn(len(x_real), 16, device=device))
        d_loss = d_loss_fn(Dd, x_real, x_fake)                  # §2 의 내 손실 그대로 / your §2 loss as is
        opt_Dd.zero_grad(); d_loss.backward(); opt_Dd.step()
        g_loss = g_loss_fn(Dd, x_fake)
        opt_Gd.zero_grad(); g_loss.backward(); opt_Gd.step()
    print(f"epoch {ep}: d_loss {d_loss.item():.3f}   g_loss {g_loss.item():.3f}   {time.time() - t0:.0f} s")

Gd.eval()                                                       # 그림을 그릴 때는 평가 모드(배치 정규화가 저장한 평균을 씀) / evaluation mode for the grid (batch norm uses stored averages)
with torch.no_grad():
    imgs = ((Gd(Z_GRID) + 1) / 2).squeeze(1).cpu()             # [-1, 1] -> [0, 1]
Gd.train()
fig, axes = plt.subplots(8, 8, figsize=(4.8, 4.8))
for k, ax in enumerate(axes.flat):
    ax.imshow(imgs[k], cmap="gray_r", vmin=0, vmax=1); ax.axis("off")
plt.suptitle(f"DCGAN after {EPOCHS} epoch(s), 64 fixed z", fontsize=10); plt.tight_layout(); plt.show()
# 기대값 / Expected
#   epoch 1: d_loss 0.313   g_loss 1.709   (2코어 CPU 약 75초 / about 75 s on a 2-core CPU)
#   숫자 모양이 나오기 시작하지만 획이 거칠거나 끊긴 것이 많습니다. 에폭을 늘리면 획이 또렷해집니다.
#   슬라이드 「Did We Get Sharper Samples?」의 오른쪽 칸은 같은 구조로 8에폭 학습한 것입니다.
#   Digit shapes begin to appear, but many strokes are rough or broken. More epochs make the strokes sharper.
#   The right panel of the slide "Did We Get Sharper Samples?" is the same structure after 8 epochs.

**What to notice.** 손실 두 줄은 고리와 똑같고, 이미지에 맞춘 합성곱 네트워크로 바꾸자 숫자 모양이 나옵니다. 이 판별자에는 배치 정규화가 있습니다. **과제 2 에서 이 구조를 WGAN-GP 비평가로 바꿀 때는 배치 정규화를 빼세요**(§3, 층 정규화를 대신 쓸 수 있음).
*The two loss lines are the same as on the ring; switching to convolutional networks shaped for images gives digit-like strokes. This discriminator has batch normalization. **When you turn this structure into a WGAN-GP critic in Assignment 2, remove the batch normalization** (§3; layer normalization can take its place).*

**과제 2 에서 이미지에 기울기 벌점을 쓸 때.** §3 의 점은 크기가 (B, 2) 였지만 이미지는 (B, 1, 28, 28) 입니다. ε 은 이미지마다 하나 `eps = torch.rand(B, 1, 1, 1)` 로 뽑아 채널 · 세로 · 가로로 퍼지게 하고, 기울기 크기는 이미지마다 펼친 뒤 `grad.flatten(start_dim=1).norm(dim=1)` 로 잽니다. §3 의 `grad.norm(dim=1)` 을 그대로 쓰면 채널 방향으로만 재서 틀립니다.
*Gradient penalty on images, for Assignment 2. The points in §3 had shape (B, 2); images have shape (B, 1, 28, 28). Draw one ε per image with `eps = torch.rand(B, 1, 1, 1)` so it spreads over channels, height and width, and measure the gradient size per image after flattening, `grad.flatten(start_dim=1).norm(dim=1)`. Reusing §3's `grad.norm(dim=1)` would measure along the channel axis only, which is wrong.*

## §5 예측 다시 읽기 / Re-read your predictions
아래를 읽기 전에 세 문장을 먼저 쓰세요. 무엇이 좋아졌나? 무엇은 좋아지지 않았나? 그 근거는 무엇인가(덮은 군집 수 · 군집 밖 비율 · 고정 z 그림 중 어느 것)?
*Before reading on, write one sentence for each: What improved? What did not improve? What evidence supports your claim (covered clusters, off-cluster share, or the fixed-z pictures)?*

1. **GAN, 시드 2, 10,000 걸음.** 이 노트북을 만든 환경에서 8개 중 4개였습니다(군집 밖 5.3%). 같은 코드, 다른 시드에서는 8 · 7 · 6 · 6(시드 0 · 1 · 3 · 4)으로, 시드마다 달랐습니다.
   *GAN, seed 2, 10,000 steps. 4 of 8 in the environment used to build this notebook (5.3% off-cluster). The same code with other seeds gave 8, 7, 6, 6 (seeds 0, 1, 3, 4): it changed from seed to seed.*
2. **손실값만으로는 믿을 만하게 알 수 없었습니다.** 두 곡선은 오르내리기만 했습니다. 고정 z 로 그린 표본을 보고, 덮은 군집 수를 세야 합니다(슬라이드 「Training Well ≠ Generating Well」).
   *Loss values alone could not reliably tell. The curves only moved up and down. Look at samples drawn from fixed z and count the covered clusters (slide "Training Well ≠ Generating Well").*
3. **WGAN-GP, 같은 시드 2.** G 1,000 걸음에 이미 8개를 덮었고(그때 군집 밖 28.0%) 점수 차가 줄었습니다. 그러나 G 3,000 걸음에서도 군집 밖 점은 11.0% 로 GAN 의 5.3% 보다 많았고, 시드 3 처럼 늦게 자리 잡는 경우도 있었습니다. 두 방법은 걸음 수와 옵티마이저 설정도 달랐습니다(§3 의 비교 조건 표). 「더 안정적인 경우가 많다」이지 「늘 이긴다」가 아닙니다.
   *WGAN-GP, the same seed 2. 8 clusters by 1,000 G steps (28.0% off-cluster at that point), with a falling score gap. But even at 3,000 G steps 11.0% of its points were off-cluster, more than the GAN's 5.3%, and some seeds, such as seed 3, settled late. The two methods also differed in step counts and optimizer settings (the comparison table in §3). "Often steadier", not "always wins".*

### 더 해 보고 싶다면 / For the curious (Optional)
- §3 셀의 `LAMBDA` 를 1 이나 100 으로, 또는 `N_CRITIC` 을 1 로 바꿔 다시 돌려 보세요. 덮는 군집 수와 점수 차 곡선이 어떻게 달라지는지 직접 보세요.
  *Change `LAMBDA` in §3 to 1 or 100, or `N_CRITIC` to 1, and run again. See for yourself how the covered count and the score-gap curve change.*
- 원 WGAN 은 벌점 대신 비평가의 가중치를 [−0.01, 0.01] 로 잘랐습니다. WGAN-GP 논문 §3 이 그 방법이 비평가를 너무 단순하게 만들고 기울기가 폭주하거나 사라지게 한다고 보였고, §5.6 은 점수 차 곡선이 표본 품질과 함께 움직였다는 것과 비평가가 과적합하면 그 관계가 깨진다는 것을 함께 적었습니다(논문 목록 C4, 7주차 발표).
  *The original WGAN clipped the critic's weights to [−0.01, 0.01] instead of using a penalty. Section 3 of the WGAN-GP paper shows that clipping makes the critic too simple and makes gradients explode or vanish; Section 5.6 reports that the score-gap curve moved with sample quality, and that this link breaks when the critic overfits (paper list C4, presented in Week 7).*

In [ ]:
# 실행 환경 기록 / Record the runtime
import sys, platform, datetime
print("Runtime :", platform.platform(), "| Python", sys.version.split()[0])
print("torch   :", torch.__version__, "| threads", torch.get_num_threads(), "|", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("Run at  :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))